# School of Thinkers – DA (Professor–Student)

Author: Luís Anunciação

Loads `../ep.rds` (DP output). DV: student self report total (mean of the 11 items). Teacher responses to the classroom inventory are joined to students by class (`demographics.class_evaluated` = student `class_id`); a class with k teacher responses duplicates its students k times. Question: which teacher variable predicts higher student scores.

In [1]:
# Setup
pacman::p_load(tidyverse, lme4, lmerTest, psych)
ep <- readRDS("../ep.rds")

### 1. Teacher scores

Practice items 5–14 (1–5), class perception items 15–25 (1–4; 0 = cannot say → NA), QoL inventory dimensions per codebook (1–5). Mental health items 1, 2, 3 and 31 are worded positively and are reversed so the score reads as distress. Item 39 (harassment) is reversed inside Relationship.

In [2]:
# Teacher frame: one row per classroom inventory response (pratica), with QoL dimension means
rev5 <- \(x) 6 - x  # reverse a 1-5 item
df_t <- ep$df_t |>
  filter(has_pratica, !is.na(demographics.class_evaluated)) |>
  mutate(across(num_range("matrix_2.", 15:25, suffix = "_num"), \(x) na_if(x, 0)),
         across(c(questions_matrix.1_num, questions_matrix.2_num, questions_matrix.3_num,
                  questions_matrix.31_num, questions_matrix.39_num), rev5)) |>
  transmute(teacher_id = respondent_id, class_id = demographics.class_evaluated,
            role = demographics.role,
            years_edu = factor(demographics.time_in_education_pratica,
                               levels = c("até 1 ano", "De 1 a 3 anos", "De 3 a 6 anos", "De 6 a 9 anos",
                                          "De 9 a 12 anos", "De 12 a 15 anos", "acima de 15 anos")) |> as.integer(),
            training_nd = initial_questions.q_1_num,
            confidence_reg = initial_questions.q_2_num,
            debates = c(Nunca. = 1, Raramente = 2, Mensalmente = 3, Semanalmente = 4)[initial_questions.q_4],
            practice = rowMeans(across(num_range("matrix_1.", 5:14, suffix = "_num")), na.rm = TRUE),
            class_perception = rowMeans(across(num_range("matrix_2.", 15:25, suffix = "_num")), na.rm = TRUE),
            distress = rowMeans(across(num_range("questions_matrix.", 1:31, suffix = "_num")), na.rm = TRUE),
            autonomy = rowMeans(across(num_range("questions_matrix.", 32:35, suffix = "_num")), na.rm = TRUE),
            relationship = rowMeans(across(num_range("questions_matrix.", 36:39, suffix = "_num")), na.rm = TRUE),
            climate = rowMeans(across(num_range("questions_matrix.", 40:43, suffix = "_num")), na.rm = TRUE),
            leadership = rowMeans(across(num_range("questions_matrix.", 44:47, suffix = "_num")), na.rm = TRUE),
            satisfaction = rowMeans(across(num_range("questions_matrix.", 48:49, suffix = "_num")), na.rm = TRUE)) |>
  mutate(across(where(is.numeric), \(x) ifelse(is.nan(x), NA, x)))
cat("teacher responses:", nrow(df_t), "| distinct classes:", n_distinct(df_t$class_id), "\n")
print(table(df_t$role, useNA = "a"))
df_t |> select(years_edu:satisfaction) |> psych::describe() |> round(2)

teacher responses: 90 | distinct classes: 61 



                                auxiliar de sala 
                                               1 
                                Auxiliar de sala 
                                               4 
                                Auxiliar de Sala 
                                               1 
                            Auxiliar de sala 30h 
                                               1 
                                     Coordenador 
                                               1 
                                         Diretor 
                                               1 
                                        Diretora 
                                               1 
                                       Professor 
                                              13 
                             Professor-Geografia 
                                               1 
                             professor de MUSICA 
                                               1 

,vars,n,mean,sd,median,trimmed,mad,min,max,range,skew,kurtosis,se
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
years_edu,1,90,4.13,1.94,4.00,4.14,2.22,1.00,7.00,6.00,0.18,-1.23,0.20
training_nd,2,90,0.34,0.48,0.00,0.31,0.00,0.00,1.00,1.00,0.64,-1.60,0.05
confidence_reg,3,90,3.33,0.96,3.00,3.33,1.48,1.00,5.00,4.00,-0.09,-0.20,0.10
debates,4,90,3.38,0.88,4.00,3.53,0.00,1.00,4.00,3.00,-1.19,0.31,0.09
practice,5,90,4.08,0.53,4.10,4.10,0.59,2.40,5.00,2.60,-0.47,-0.06,0.06
class_perception,6,90,3.13,0.44,3.09,3.13,0.40,2.00,4.00,2.00,-0.01,-0.21,0.05
distress,7,81,2.61,0.58,2.68,2.61,0.62,1.06,4.03,2.97,0.04,-0.39,0.06
autonomy,8,79,3.98,0.67,4.00,4.02,0.37,2.00,5.00,3.00,-0.53,0.35,0.08
relationship,9,79,4.11,0.78,4.25,4.21,0.37,1.50,5.00,3.50,-1.23,1.26,0.09


### 2. Merge

Students (n = 741, 40 classes) joined to teacher responses by class. Students in classes without a teacher response are dropped.

In [3]:
# df: one row per student x teacher response
df <- ep$df_st |>
  transmute(student_id = respondent_id, class_id, school = school_name,
            sex = demographics.gender, age = demographics.age_num,
            score = rowMeans(across(num_range("behavior_matrix.", 1:11, suffix = "_num")))) |>
  inner_join(df_t, by = "class_id", relationship = "many-to-many")
cat("rows:", nrow(df), "| students:", n_distinct(df$student_id), "| classes:", n_distinct(df$class_id),
    "| teacher responses:", n_distinct(df$teacher_id), "\n")
cat("students dropped (no teacher response):", n_distinct(ep$df_st$respondent_id) - n_distinct(df$student_id), "\n")
print(table(teacher_responses_per_class = table(distinct(df, class_id, teacher_id)$class_id)))
cat("\nclass level ICC of the student score (students nested in classes):\n")
lmer(score ~ 1 + (1 | class_id), data = distinct(df, student_id, .keep_all = TRUE)) |>
  VarCorr() |> as.data.frame() |> (\(v) v$vcov[1] / sum(v$vcov))() |> round(3)

rows: 782 | students: 606 | classes: 34 | teacher responses: 46 


students dropped (no teacher response): 135 


teacher_responses_per_class
 1  2  3  4 
26  5  2  1 



class level ICC of the student score (students nested in classes):


boundary (singular) fit: see help('isSingular')



[1] 0

### 3. Teacher predictors of the student score

One mixed model per teacher variable: `score ~ x + school + (1 | class_id)`, rows weighted by 1/k (k = teacher responses in the class) so each student carries total weight 1 despite duplication. Predictors stay in their raw metric; `b` is the change in the student score (1–5) per unit of the teacher variable.

In [4]:
# fit_one(x): mixed model for one teacher predictor; returns b, SE, t, df, p and n
fit_one <- \(x) lmer(reformulate(c(x, "school", "(1 | class_id)"), "score"), weights = w,
                     data = drop_na(df, all_of(x)) |> mutate(w = 1 / n_distinct(teacher_id), .by = class_id), REML = TRUE) |>
  (\(m) summary(m)$coefficients[x, , drop = FALSE] |> as_tibble() |>
     mutate(predictor = x, n_rows = nobs(m), .before = 1))()
c("years_edu", "training_nd", "confidence_reg", "debates", "practice", "class_perception",
  "distress", "autonomy", "relationship", "climate", "leadership", "satisfaction") |>
  map(fit_one) |> list_rbind() |>
  rename(b = Estimate, SE = `Std. Error`, t = `t value`, p = `Pr(>|t|)`) |>
  mutate(across(c(b, SE, t, df), \(v) round(v, 3)), p = round(p, 4)) |>
  arrange(p)

predictor,n_rows,b,SE,df,t,p
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
relationship,681,-0.061,0.036,27.265,-1.718,0.0970
climate,681,-0.063,0.038,25.032,-1.657,0.1100
autonomy,681,-0.070,0.045,38.552,-1.538,0.1322
distress,683,0.068,0.047,31.934,1.441,0.1594
class_perception,782,-0.089,0.063,33.702,-1.420,0.1649
leadership,681,-0.024,0.031,21.654,-0.781,0.4435
satisfaction,681,-0.023,0.031,27.778,-0.725,0.4743
confidence_reg,782,-0.013,0.023,25.081,-0.575,0.5703
training_nd,782,-0.015,0.060,32.492,-0.255,0.8000


### 4. Check: class means

Same question at the class level (one row per teacher response, class mean score as DV), which removes the duplication instead of modelling it.

In [5]:
# Class level: correlation of each teacher variable with the class mean student score
df |>
  summarise(class_score = mean(score), n_students = n_distinct(student_id),
            across(years_edu:satisfaction, first), .by = c(class_id, teacher_id)) |>
  (\(d) map(c("years_edu", "training_nd", "confidence_reg", "debates", "practice", "class_perception",
              "distress", "autonomy", "relationship", "climate", "leadership", "satisfaction"),
            \(x) cor.test(d[[x]], d$class_score, use = "complete.obs") |>
              (\(ct) tibble(predictor = x, n = ct$parameter + 2, r = round(ct$estimate, 3), p = round(ct$p.value, 4)))()) |>
     list_rbind())() |>
  arrange(p)

predictor,n,r,p
<chr>,<dbl>,<dbl>,<dbl>
confidence_reg,46,-0.170,0.2580
class_perception,46,-0.116,0.4409
training_nd,46,0.077,0.6097
satisfaction,39,0.060,0.7171
distress,40,-0.053,0.7433
leadership,39,0.052,0.7516
debates,46,0.034,0.8242
relationship,39,-0.035,0.8312
practice,46,0.029,0.8459
